In [1]:
import duckdb
import pandas as pd

con = duckdb.connect("../telco.duckdb")

df = con.execute("SELECT * FROM vw_customer_value_segments").df()

print(df.shape)
df.head()

(7043, 18)


,customer_id,tenure_months,monthly_charges,total_charges,contract,internet_service,churn_value,tenure_band,has_internet,has_phone,addon_count,contract_risk,automatic_payment,spend_band,expected_remaining_months,historic_margin,clv,value_decile
0,7569-NMZYQ,72,118.75,8672.45,Two year,Fiber optic,0,Loyal,1,1,6,1,1,High,36,2601.74,1282.50,1
1,8984-HPEMB,71,118.65,8477.60,Two year,Fiber optic,0,Loyal,1,1,6,1,0,High,36,2543.28,1281.42,1
2,5989-AXPUC,68,118.60,7990.05,Two year,Fiber optic,0,Loyal,1,1,6,1,0,High,36,2397.02,1280.88,1
3,9924-JPRMC,72,118.20,8547.15,Two year,Fiber optic,0,Loyal,1,1,6,1,0,High,36,2564.15,1276.56,1
4,3810-DVDQQ,72,117.60,8308.90,Two year,Fiber optic,0,Loyal,1,1,6,1,1,High,36,2492.67,1270.08,1


In [2]:
target = "churn_value"

features = [
    "tenure_months",
    "monthly_charges",
    "total_charges",
    "has_phone",
    "addon_count",
    "contract_risk",
    "automatic_payment",
    "internet_service",
]

X = pd.get_dummies(df[features], columns=["internet_service"], drop_first=True)
y = df[target]

print(X.shape)
X.head()

(7043, 9)


,tenure_months,monthly_charges,total_charges,has_phone,addon_count,contract_risk,automatic_payment,internet_service_Fiber optic,internet_service_No
0,72,118.75,8672.45,1,6,1,1,True,False
1,71,118.65,8477.60,1,6,1,0,True,False
2,68,118.60,7990.05,1,6,1,0,True,False
3,72,118.20,8547.15,1,6,1,0,True,False
4,72,117.60,8308.90,1,6,1,1,True,False


In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.25,
    stratify=y,
    random_state=42
)

print(X_train.shape, X_test.shape)
print(y_train.mean(), y_test.mean())

(5282, 9) (1761, 9)
0.2654297614539947 0.26519023282226006


In [4]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

logreg = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])

logreg.fit(X_train, y_train)
print("trained")

trained


In [5]:
from sklearn.metrics import classification_report, confusion_matrix, roc_auc_score

y_pred  = logreg.predict(X_test)
y_proba = logreg.predict_proba(X_test)[:, 1]

print(confusion_matrix(y_test, y_pred))
print()
print(classification_report(y_test, y_pred, digits=3))
print("ROC AUC:", round(roc_auc_score(y_test, y_proba), 3))

[[932 362]
 [ 97 370]]

              precision    recall  f1-score   support

           0      0.906     0.720     0.802      1294
           1      0.505     0.792     0.617       467

    accuracy                          0.739      1761
   macro avg      0.706     0.756     0.710      1761
weighted avg      0.800     0.739     0.753      1761

ROC AUC: 0.84


In [6]:
coefs = pd.DataFrame({
    "feature": X_train.columns,
    "coefficient": logreg.named_steps["model"].coef_[0]
}).sort_values("coefficient", ascending=False)

coefs

,feature,coefficient
1,monthly_charges,1.835196
5,contract_risk,0.733298
2,total_charges,0.573623
8,internet_service_No,0.263224
6,automatic_payment,-0.172772
7,internet_service_Fiber optic,-0.269150
3,has_phone,-0.594036
4,addon_count,-0.821713
0,tenure_months,-1.218183


In [7]:
features_v2 = [f for f in X.columns if f != "total_charges"]

X_train2, X_test2 = X_train[features_v2], X_test[features_v2]

logreg2 = Pipeline([
    ("scale", StandardScaler()),
    ("model", LogisticRegression(max_iter=1000, class_weight="balanced"))
])
logreg2.fit(X_train2, y_train)

print("ROC AUC:", round(roc_auc_score(y_test, logreg2.predict_proba(X_test2)[:,1]), 3))

pd.DataFrame({
    "feature": X_train2.columns,
    "coefficient": logreg2.named_steps["model"].coef_[0]
}).sort_values("coefficient", ascending=False)


ROC AUC: 0.838


,feature,coefficient
1,monthly_charges,2.027357
4,contract_risk,0.727646
7,internet_service_No,0.328214
5,automatic_payment,-0.180307
6,internet_service_Fiber optic,-0.275306
2,has_phone,-0.588961
0,tenure_months,-0.742260
3,addon_count,-0.797284


In [8]:
con.execute("""
    SELECT internet_service, COUNT(*) AS customers, ROUND(AVG(churn_value),3) AS churn_rate
    FROM vw_customer_features GROUP BY 1 ORDER BY 3 DESC
""").df()

,internet_service,customers,churn_rate
0,Fiber optic,3096,0.419
1,DSL,2421,0.190
2,No,1526,0.074


In [9]:
from sklearn.ensemble import HistGradientBoostingClassifier

gb = HistGradientBoostingClassifier(random_state=42)
gb.fit(X_train2, y_train)

y_pred_gb  = gb.predict(X_test2)
y_proba_gb = gb.predict_proba(X_test2)[:, 1]

print(classification_report(y_test, y_pred_gb, digits=3))
print("ROC AUC:", round(roc_auc_score(y_test, y_proba_gb), 3))

              precision    recall  f1-score   support

           0      0.828     0.894     0.860      1294
           1      0.624     0.486     0.546       467

    accuracy                          0.786      1761
   macro avg      0.726     0.690     0.703      1761
weighted avg      0.774     0.786     0.777      1761

ROC AUC: 0.829


In [10]:
gb = HistGradientBoostingClassifier(random_state=42, class_weight="balanced")
gb.fit(X_train2, y_train)

y_pred_gb  = gb.predict(X_test2)
y_proba_gb = gb.predict_proba(X_test2)[:, 1]

print(classification_report(y_test, y_pred_gb, digits=3))
print("ROC AUC:", round(roc_auc_score(y_test, y_proba_gb), 3))

              precision    recall  f1-score   support

           0      0.895     0.733     0.806      1294
           1      0.507     0.762     0.609       467

    accuracy                          0.740      1761
   macro avg      0.701     0.747     0.707      1761
weighted avg      0.792     0.740     0.754      1761

ROC AUC: 0.833
